# Evaluación held-out del predictor lineal identidad

Este notebook ejecutará la primera evaluación en test del smoke experiment del paper. Antes de construir test, reproduce con train/validation los checkpoints congelados para seeds `5–9` y aborta si cualquier epoch, estado o métrica no coincide.

La configuración, las métricas y todos los umbrales están congelados en `configs/paper_linear_identity_heldout_smoke.yaml`. Este archivo se prepara y revisa sin outputs antes de su primera ejecución.

## Alcance y regla de decisión

Ésta es una prueba held-out **pequeña**: 32/8/8 masters por régimen para train/validation/test. No es todavía la escala publicada de 7.000/2.000/1.000 por régimen.

Cada una de las cinco seeds debe satisfacer simultáneamente: error relativo a identidad ≤ 5%, norma antisimétrica relativa ≤ 5%, error medio de acción sobre centroides K-means ≤ 2%, al menos 18 autovalores con `|λ−1|≤0.05`, rango efectivo de test ≥ 4 y valores finitos. K-means usa `K=18`, `n_init=20` y seed 0 sobre embeddings crudos del encoder online.

El resultado del gate es la conjunción de todas las seeds; ninguna media puede ocultar una corrida fallida. Una vez ejecutado este notebook, test queda consumido para este protocolo y los umbrales no pueden ajustarse retrospectivamente.

In [ ]:
# ruff: noqa: E402, E501
import json
import platform
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "No se encontró la raíz del repositorio."
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import load_paper_linear_identity_heldout_config
from koopman_jepa.paper_data import (
    PAPER_REGIME_NAMES,
    PaperRegimeDataset,
    generate_paper_master,
)
from koopman_jepa.paper_evaluation import (
    evaluate_paper_linear_identity,
    evaluate_paper_linear_identity_heldout_gate,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
config_path = ROOT / "configs" / "paper_linear_identity_heldout_smoke.yaml"
config = load_paper_linear_identity_heldout_config(config_path)
torch.use_deterministic_algorithms(True)

print(config_path.relative_to(ROOT))
print(json.dumps(asdict(config), indent=2))

## Barrera previa: reproducir checkpoints sin test

Las siguientes celdas sólo construyen train y validation. Cada seed vuelve a entrenarse, captura el estado de todos los epochs y recarga el checkpoint seleccionado. Test no existe todavía en memoria.

In [ ]:
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {
    validation_dataset.sample_key(index) for index in range(len(validation_dataset))
}

assert len(train_dataset) == 576
assert len(validation_dataset) == 144
assert train_keys.isdisjoint(validation_keys)
print(f"Train/validation: {len(train_dataset)}/{len(validation_dataset)}")
print("Intersección train/validation: 0 — PASS")
print("Test todavía no fue instanciado.")

In [ ]:
expected_epochs = dict(
    zip(config.sweep.seeds, config.replay.expected_epochs, strict=True)
)
models = {}
checkpoint_runs = {}
replay_results = {}
replay_times = {}

for seed in config.sweep.seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    run_config = replace(config.train, seed=seed)
    model = PaperTemporalJEPA(config.model)

    started = time.perf_counter()
    checkpoint_run = run_paper_train_validation_with_checkpoint(
        model,
        train_dataset,
        validation_dataset,
        run_config,
        config.checkpoint_gate,
    )
    replay = verify_paper_checkpoint_replay(
        model,
        checkpoint_run,
        validation_dataset,
        run_config,
        config.replay,
        expected_epoch=expected_epochs[seed],
    )
    replay_times[seed] = time.perf_counter() - started
    models[seed] = model
    checkpoint_runs[seed] = checkpoint_run
    replay_results[seed] = replay

    selected_epoch = (
        checkpoint_run.selection.epoch
        if checkpoint_run.selection is not None
        else None
    )
    maximum_metric_error = max(
        replay.validation_loss_absolute_error,
        replay.validation_embedding_std_absolute_error,
        replay.validation_effective_rank_absolute_error,
    )
    print(
        f"seed={seed} expected={expected_epochs[seed]} selected={selected_epoch} "
        f"max_replay_error={maximum_metric_error:.3g} "
        f"status={'PASS' if replay.passed else 'FAIL'} "
        f"time={replay_times[seed]:.2f}s"
    )

replay_gate_open = (
    set(replay_results) == set(config.sweep.seeds)
    and all(result.passed for result in replay_results.values())
)
assert replay_gate_open, (
    "ABORTAR: al menos un checkpoint no reprodujo el epoch o las métricas "
    "congeladas. Test no debe construirse."
)
print(f"Replay gate: PASS; tiempo total={sum(replay_times.values()):.2f}s")
print("La próxima celda es la primera que puede construir test.")

## Apertura held-out

Esta frontera es intencional. Si el replay anterior falla, la ejecución se detiene y esta sección no corre. Si pasa, test se construye una sola vez bajo las reglas ya congeladas.

In [ ]:
assert globals().get("replay_gate_open", False), (
    "El replay gate no está abierto; no se permite construir test."
)
test_dataset = PaperRegimeDataset(config.data, "test", generate_paper_master)
test_keys = {test_dataset.sample_key(index) for index in range(len(test_dataset))}

assert len(test_dataset) == 8 * len(PAPER_REGIME_NAMES) == 144
assert train_keys.isdisjoint(test_keys)
assert validation_keys.isdisjoint(test_keys)
print(f"Test construido: {len(test_dataset)} secuencias")
print("Intersecciones con train y validation: 0 — PASS")

In [ ]:
@torch.no_grad()
def collect_online_embeddings(model, dataset, run_config):
    device = torch.device(run_config.device)
    model.to(device)
    model.eval()
    loader = make_paper_loader(dataset, run_config, shuffle=False)
    chunks = []
    for context, _, _ in loader:
        chunks.append(model.online_encoder(context.to(device)).cpu().numpy())
    return np.concatenate(chunks, axis=0)


metrics_by_seed = {}
test_embeddings_by_seed = {}
for seed in config.sweep.seeds:
    run_config = replace(config.train, seed=seed)
    model = models[seed]
    embeddings = collect_online_embeddings(model, test_dataset, run_config)
    matrix = model.predictor.matrix.detach().cpu().numpy()
    metrics = evaluate_paper_linear_identity(
        matrix,
        embeddings,
        config.evaluation,
    )
    test_embeddings_by_seed[seed] = embeddings
    metrics_by_seed[seed] = metrics

gate = evaluate_paper_linear_identity_heldout_gate(
    metrics_by_seed,
    config.sweep,
    config.gate,
)
print(json.dumps(asdict(gate), indent=2))

In [ ]:
print("seed  id_error  skew_norm  centroid_action  eig_near_1  test_rank")
for seed, metrics in metrics_by_seed.items():
    print(
        f"{seed:>4d}  {metrics.relative_identity_error:>8.4%}  "
        f"{metrics.relative_skew_norm:>9.4%}  "
        f"{metrics.mean_centroid_action_error:>15.4%}  "
        f"{metrics.near_identity_eigenvalues:>10d}  "
        f"{metrics.test_effective_rank:>9.3f}"
    )

## Diagnósticos por seed

Las líneas rojas marcan los límites congelados. En los tres primeros paneles, menor es mejor; en conteo espectral y rango efectivo, mayor es mejor. El último panel muestra los 32 autovalores de cada predictor en el plano complejo y el círculo `|λ−1|=0.05`.

In [ ]:
seeds = np.array(config.sweep.seeds)
identity_errors = np.array([metrics_by_seed[seed].relative_identity_error for seed in seeds])
skew_norms = np.array([metrics_by_seed[seed].relative_skew_norm for seed in seeds])
centroid_errors = np.array([metrics_by_seed[seed].mean_centroid_action_error for seed in seeds])
near_counts = np.array([metrics_by_seed[seed].near_identity_eigenvalues for seed in seeds])
effective_ranks = np.array([metrics_by_seed[seed].test_effective_rank for seed in seeds])

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
bar_specs = [
    (axes[0, 0], identity_errors, config.gate.max_relative_identity_error, "Error relativo a identidad", "≤"),
    (axes[0, 1], skew_norms, config.gate.max_relative_skew_norm, "Norma antisimétrica relativa", "≤"),
    (axes[0, 2], centroid_errors, config.gate.max_mean_centroid_action_error, "Error de acción en centroides", "≤"),
    (axes[1, 0], near_counts, config.gate.min_near_identity_eigenvalues, "Autovalores cerca de 1", "≥"),
    (axes[1, 1], effective_ranks, config.gate.min_test_effective_rank, "Rango efectivo de test", "≥"),
]
for axis, values, threshold, title, direction in bar_specs:
    axis.bar(seeds, values, color="tab:blue", alpha=0.8)
    axis.axhline(threshold, color="tab:red", linestyle="--", label=f"límite {direction} {threshold:g}")
    axis.set(title=title, xlabel="Seed")
    axis.legend()

theta = np.linspace(0.0, 2.0 * np.pi, 300)
axes[1, 2].plot(
    1.0 + config.evaluation.eigenvalue_tolerance * np.cos(theta),
    config.evaluation.eigenvalue_tolerance * np.sin(theta),
    color="tab:red",
    linestyle="--",
    label="tolerancia",
)
for seed in seeds:
    metrics = metrics_by_seed[seed]
    axes[1, 2].scatter(
        metrics.eigenvalue_real,
        metrics.eigenvalue_imag,
        s=22,
        alpha=0.65,
        label=f"seed {seed}",
    )
axes[1, 2].axhline(0.0, color="0.7", linewidth=1)
axes[1, 2].axvline(1.0, color="0.7", linewidth=1)
axes[1, 2].set(
    title="Espectro completo del predictor",
    xlabel="Parte real",
    ylabel="Parte imaginaria",
)
axes[1, 2].set_aspect("equal", adjustable="datalim")
axes[1, 2].legend(ncol=2, fontsize=8)

plt.show()

In [ ]:
status = "PASS" if gate.passed else "FAIL"
criteria = {
    "cinco seeds presentes": gate.all_seeds_present,
    "métricas finitas": gate.all_finite,
    "distancia a identidad": gate.identity_passed,
    "simetría": gate.skew_passed,
    "acción sobre centroides": gate.centroid_action_passed,
    "concentración espectral": gate.eigenvalues_passed,
    "rango efectivo": gate.effective_rank_passed,
}
failed_criteria = [name for name, passed in criteria.items() if not passed]
failed_text = ", ".join(failed_criteria) if failed_criteria else "ninguno"
failed_seed_text = ", ".join(str(seed) for seed in gate.failed_seeds) or "ninguna"

identity_mean, identity_std = identity_errors.mean(), identity_errors.std()
skew_mean, skew_std = skew_norms.mean(), skew_norms.std()
centroid_mean, centroid_std = centroid_errors.mean(), centroid_errors.std()
rank_mean, rank_std = effective_ranks.mean(), effective_ranks.std()

pressures = {
    "identidad": gate.worst_relative_identity_error / config.gate.max_relative_identity_error,
    "simetría": gate.worst_relative_skew_norm / config.gate.max_relative_skew_norm,
    "acción en centroides": gate.worst_mean_centroid_action_error / config.gate.max_mean_centroid_action_error,
    "conteo espectral": config.gate.min_near_identity_eigenvalues / max(gate.minimum_near_identity_eigenvalues, 1),
    "rango efectivo": config.gate.min_test_effective_rank / max(gate.minimum_test_effective_rank, 1e-12),
}
closest_criterion = max(pressures, key=pressures.get)

decision = (
    "El smoke held-out lineal-identidad pasa en las cinco seeds. El siguiente paso científico es decidir entre escalar esta condición al dataset completo o ejecutar primero el control de inicialización aleatoria; no corresponde retocar este test ya consumido."
    if gate.passed
    else "El smoke held-out lineal-identidad falla. Debemos reportar el fallo y diagnosticarlo sin cambiar checkpoints, umbrales ni esta partición de test."
)

display(Markdown(f"""## Resultado e interpretación

- **Gate global: {status}.**
- **Criterios fallidos:** {failed_text}.
- **Seeds fallidas:** {failed_seed_text}.
- **Peor error a identidad:** `{gate.worst_relative_identity_error:.3%}` frente al máximo `{config.gate.max_relative_identity_error:.1%}`.
- **Peor norma antisimétrica:** `{gate.worst_relative_skew_norm:.3%}` frente al máximo `{config.gate.max_relative_skew_norm:.1%}`.
- **Peor acción sobre centroides:** `{gate.worst_mean_centroid_action_error:.3%}` frente al máximo `{config.gate.max_mean_centroid_action_error:.1%}`.
- **Mínimo conteo cerca de 1:** `{gate.minimum_near_identity_eigenvalues}` frente al mínimo `{config.gate.min_near_identity_eigenvalues}`.
- **Mínimo rango efectivo:** `{gate.minimum_test_effective_rank:.3f}` frente al mínimo `{config.gate.min_test_effective_rank:.1f}`.

### Lectura de los gráficos

Las barras muestran la variabilidad seed por seed y evitan que una media favorable esconda una corrida problemática. El criterio relativamente más cercano a su frontera es **{closest_criterion}**. El plano complejo permite distinguir una matriz realmente cercana a identidad de una que sólo tiene algunos autovalores correctos; el conteo espectral por sí solo no controla no-normalidad ni acción sobre los centroides.

Las medias ± desviación poblacional son: identidad `{identity_mean:.3%} ± {identity_std:.3%}`, antisimetría `{skew_mean:.3%} ± {skew_std:.3%}`, acción en centroides `{centroid_mean:.3%} ± {centroid_std:.3%}` y rango efectivo `{rank_mean:.2f} ± {rank_std:.2f}`. Como referencia descriptiva, el paper informa 2.34%, 2.06% y 0.80% para las tres primeras cantidades, pero no publica seeds ni dispersión suficientes para una prueba formal de equivalencia.

### Decisión

{decision}

Este resultado usa sólo 144 secuencias de test y el modelo lineal-identidad. No reproduce la pureza 65.48% del predictor MLP, no incluye el autoencoder y no constituye todavía una reproducción paper-scale.
"""))